## Operators in MongoDB

Disclaimer: the following table doesn't imply any horizontal relationship between "first level" and "second level" operators. 

|"First level"|"Second level"|
|---|---|
|match|elemMatch, type, size|
|group|sum, avg, max, min|
|unwind|push|
|sort|first, last|
|project|gte, gt, lte, lt|
|limit|and, or, cond|
|replaceRoot|mergeObjects|

Let's re-define a function to perform queries by PyMongo using ``aggregate`` method.

### `match`
*   **What it is:** A **filter**. It's like the `WHERE` clause in SQL.
*   **What it does:** It looks at all your documents and only keeps the ones that meet the conditions you set. You can find the documentation for [`$match`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/match/) here.
*   **Second Level (Tools for filtering):**
    *   [`$elemMatch`](https://www.mongodb.com/docs/manual/reference/operator/query/elemMatch/): Finds documents where at least one item in a list (array) matches several criteria at once.
    *   [`$type`](https://www.mongodb.com/docs/manual/reference/operator/query/type/): Checks if a field's value is a certain data type (like a string, number, or array).
    *   [`$size`](https://www.mongodb.com/docs/manual/reference/operator/query/size/): Checks if a list (array) has a specific number of items.

### `group`
*   **What it is:** A **grouper** or a **bucket sorter**.
*   **What it does:** It groups multiple documents together into a single "bucket" based on a shared value (e.g., group all sales by `user_id`). You can then perform calculations on each group. You can find the documentation for [`$group`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/group/) here.
*   **Second Level (Tools for calculating within groups):**
    *   [`$sum`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/sum/): Adds up all the numeric values for a field within the group.
    *   [`$avg`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/avg/): Calculates the average of the numeric values in the group.
    *   [`$max`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/max/) / [`$min`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/min/): Finds the highest or lowest value in the group.

### `unwind`
*   **What it is:** An **unpacker**.
*   **What it does:** If you have a document with a list of items (like an order with a list of products), `$unwind` will create a separate copy of that document for each item in the list. You can find the documentation for [`$unwind`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/unwind/) here.
*   **Second Level (A related but opposite tool):**
    *   [`$push`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/push/): This is the opposite of `$unwind`. It is used inside a `$group` stage to take values from multiple documents and put them together into a new list.

### `sort`
*   **What it is:** A **sorter**.
*   **What it does:** It arranges your documents in a specific order, either ascending (1 to 10, A to Z) or descending (10 to 1, Z to A). You can find the documentation for [`$sort`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/sort/) here.
*   **Second Level (Tools used with sorting in a `$group` stage):**
    *   [`$first`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/first/) / [`$last`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/last/): After documents are sorted, these operators (used inside `$group`) let you grab a value from the very first or very last document in that group.

### `project`
*   **What it is:** A **reshaper**.
*   **What it does:** It lets you choose exactly which fields from a document you want to keep, remove, or rename. You can also use it to create new fields based on calculations. You can find the documentation for [`$project`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/project/) here.
*   **Second Level (Tools for comparing values):**
    *   [`$gte`](https://www.mongodb.com/docs/manual/reference/operator/query/gte/) / [`$gt`](https://www.mongodb.com/docs/manual/reference/operator/query/gt/): "Greater than or equal to" and "Greater than."
    *   [`$lte`](https://www.mongodb.com/docs/manual/reference/operator/query/lte/) / [`$lt`](https://www.mongodb.com/docs/manual/reference/operator/query/lt/): "Less than or equal to" and "Less than." These are used to compare values, often inside a `$match` or `$project` stage.

### `limit`
*   **What it is:** A **stopper**.
*   **What it does:** It stops the pipeline from processing any more documents after a certain number has passed through. It's almost always used right after `$sort` to get the "Top 10" or "Bottom 5" results. You can find the documentation for [`$limit`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/limit/) here.
*   **Second Level (Tools for logic):**
    *   [`$and`](https://www.mongodb.com/docs/manual/reference/operator/query/and/) / [`$or`](https://www.mongodb.com/docs/manual/reference/operator/query/or/): These let you combine multiple conditions. For example, find documents where `price` is over 10 **AND** `category` is "books". (Mostly used in `$match`).
    *   [`$cond`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/cond/): An "if-then-else" operator. It checks a condition and returns one value if it's true, and another if it's false. (Mostly used in `$project`).

### `replaceRoot`
*   **What it is:** A **promoter**.
*   **What it does:** It replaces the entire document with one of its sub-documents. This is extremely useful after you `$unwind` a list of products, as it lets you "promote" each product to be the main document. You can find the documentation for [`$replaceRoot`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/replaceRoot/) here.
*   **Second Level (A tool for combining):**
    *   [`$mergeObjects`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/mergeObjects/): Takes multiple small documents (objects) and combines all their fields into a single, larger one. This is the main tool used inside `$replaceRoot`.

If you just need to find documents, use find().
If you need to calculate anything about those documents, use aggregate().

In [1]:
import pandas as pd
import pymongo
import seaborn as sns

def query(database, collection, pipeline, host='localhost', port=27017):
    try:
        client = pymongo.MongoClient(host=host, port=port)
        db = client[database]
        cll = db[collection]
    except (Exception) as e:
        print(e)
    
    else:
        try:
            cursor = cll.aggregate(pipeline)
            return pd.DataFrame(list(cursor))
        except (Exception) as e:
            print(e)
        finally:
            client.close()            

NOTE: opening and closing a connection with ``MongoClient`` everytime we need to perform a query is not efficient.
> POSSIBLE SOLUTION: define two different functions.

In [9]:
def mongo_connect(database, collection, host = "localhost", port = 27017):
    try:
        client = pymongo.MongoClient(host=host, port=port)
        db = client[database]
        cll = db[collection]
        return cll, client
    except (Exception) as e:
        print(e)

def query(cll, pipeline):
    
    try:
        cursor = cll.aggregate(pipeline, allowDiskUse = True)
        return pd.DataFrame(list(cursor))
    except (Exception) as e:
        print(e)

The first function is easy to understand but slow. The second, two-function approach is the professional and much more efficient way to work with databases, because you connect once, query many times, and disconnect once.

In [10]:
# My parameters
database = 'instacart'
collection = 'orders'

In [11]:
cll, client = mongo_connect(database, collection)
pipeline =[
    {"$match": {"order_dow":2, "order_hour_of_day":{"$gte":20}}},
    {"$project": {"_id": 0, "products":0}}
]

In [5]:
query(cll, pipeline)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2111787,41,2,2,20,24.0
1,1899315,47,5,2,22,14.0
2,1102893,54,51,2,23,2.0
3,1456703,54,73,2,23,1.0
4,1325316,54,78,2,23,1.0
...,...,...,...,...,...,...
11419,237799,60023,13,2,20,4.0
11420,1891538,60043,24,2,20,9.0
11421,1864445,60049,31,2,20,11.0
11422,2779687,60052,54,2,21,2.0


In [6]:
client.close()

### Exercise 2.1
##### Retrieve the 10 aisles with most purchases at nighttime (from 9 pm to 6 am) and compare them with the aisles having the highest number of purchases at daytime

Usually we have this order:

Start with $match

Filter as early as possible = fewer documents to process later = faster.

Then often $project

Keep only the fields you need, or create new ones.

Then do heavy stuff:

$group, $unwind, $lookup, $sort

Finish with $sort, $limit, $skip

Sorting and limiting at the end to prepare final result.

In [ ]:
cll, client = mongo_connect(database, collection)

In [ ]:
{
  "_id": {
    "$oid": "60af57782fbbe971834ae2a6"
  },
  "order_id": 2539329,
  "user_id": 1,
  "order_number": 1,
  "order_dow": 2,
  "order_hour_of_day": 8,
  "products": [
    {
      "product_id": 196,
      "product_name": "Soda",
      "aisle_id": 77,
      "aisle": "soft drinks",
      "department_id": 7,
      "department": "beverages",
      "add_to_cart_order": 1
    },
    {
      "product_id": 12427,
      "product_name": "Original Beef Jerky",
      "aisle_id": 23,
      "aisle": "popcorn jerky",
      "department_id": 19,
      "department": "snacks",
      "add_to_cart_order": 3
    },
    {
      "product_id": 14084,
      "product_name": "Organic Unsweetened Vanilla Almond Milk",
      "aisle_id": 91,
      "aisle": "soy lactosefree",
      "department_id": 16,
      "department": "dairy eggs",
      "add_to_cart_order": 2
    },
    {
      "product_id": 26088,
      "product_name": "Aged White Cheddar Popcorn",
      "aisle_id": 23,
      "aisle": "popcorn jerky",
      "department_id": 19,
      "department": "snacks",
      "add_to_cart_order": 4
    },
    {
      "product_id": 26405,
      "product_name": "XL Pick-A-Size Paper Towel Rolls",
      "aisle_id": 54,
      "aisle": "paper goods",
      "department_id": 17,
      "department": "household",
      "add_to_cart_order": 5
    }
  ]
}

In [ ]:
{
  "_id": { "$oid": "60af57782fbbe971834ae2a6" },
  "order_id": 2539329,
  "user_id": 1,
  "order_number": 1,
  "order_dow": 2,
  "order_hour_of_day": 8,
  "products": {
    "product_id": 196,
    "product_name": "Soda",
    "aisle_id": 77,
    "aisle": "soft drinks",
    "department_id": 7,
    "department": "beverages",
    "add_to_cart_order": 1
  }
}


{
  "_id": { "$oid": "60af57782fbbe971834ae2a6" },
  "order_id": 2539329,
  "user_id": 1,
  "order_number": 1,
  "order_dow": 2,
  "order_hour_of_day": 8,
  "products": {
    "product_id": 12427,
    "product_name": "Original Beef Jerky",
    "aisle_id": 23,
    "aisle": "popcorn jerky",
    "department_id": 19,
    "department": "snacks",
    "add_to_cart_order": 3
  }
}

...

In [ ]:
night_pipeline = [
    {"$match": {"$or":[{"order_hour_of_day":{"$gte":21}}, {"order_hour_of_day":{"$lte":6}}]}},
    {"$unwind": "$products"},
    {"$group": {"_id": "$products.aisle", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]

In [ ]:
day_pipeline = [
    {"$match": {"order_hour_of_day":{"$lt":21, "$gt":6}}},
    {"$unwind": "$products"},
    {"$group": {"_id": "$products.aisle", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]

In [ ]:
night_aisles = query(cll, night_pipeline)
day_aisles = query(cll, day_pipeline)

In [ ]:
pd.merge(night_aisles, day_aisles, left_index = True, right_index = True, how = "inner")

The same query in one aggregation pipeline?

In [ ]:
combined_pipeline = [
    {
        "$facet": {
            "night_aisles": [
                { "$match": { "$or": [{"order_hour_of_day": {"$gte": 21}}, {"order_hour_of_day": {"$lte": 6}}] } },
                { "$unwind": "$products" },
                { "$group": { "_id": "$products.aisle", "count": { "$sum": 1 } } },
                { "$sort": { "count": -1 } },
                { "$limit": 10 }
            ],
            "day_aisles": [
                { "$match": { "order_hour_of_day": { "$lt": 21, "$gt": 6 } } },
                { "$unwind": "$products" },
                { "$group": { "_id": "$products.aisle", "count": { "$sum": 1 } } },
                { "$sort": { "count": -1 } },
                { "$limit": 10 }
            ]
        }
    }
]

combined_results = query(cll, combined_pipeline)

In [ ]:
# Get the list of results for night and day from the first row
night_list = combined_results.loc[0, 'night_aisles']
day_list = combined_results.loc[0, 'day_aisles']

# Convert each list into a DataFrame
night_df = pd.DataFrame(night_list)
day_df = pd.DataFrame(day_list)

final_table = pd.merge(night_df, day_df, left_index=True, right_index=True, how="inner")

final_table

### Digression: Replace Root

https://www.mongodb.com/docs/manual/reference/operator/aggregation/replaceRoot/

In [12]:
match = {"$match": {"order_id": 473748}}
unwind = {"$unwind": "$products"}

# replace root
replaceRoot = {"$replaceRoot": {"newRoot": {"$mergeObjects": [{"_id": "$_id",'order_id': "$order_id"},"$products"]}}}

df_with_replace = query(cll, [match, unwind, replaceRoot])
df_with_replace.head()

,_id,order_id,product_id,product_name,aisle_id,aisle,department_id,department,add_to_cart_order,reordered
0,60af5b022fbbe97183553d36,473748,6188,Organic Apple & Butternut Squash Baby Food,92,baby food formula,18,babies,10,NaN
1,60af5b022fbbe97183553d36,473748,6347,Unsweetened Almond Milk,91,soy lactosefree,16,dairy eggs,14,NaN
2,60af5b022fbbe97183553d36,473748,7051,"Happy Tot Banana, Peach, Prune & Coconut Organ...",92,baby food formula,18,babies,2,NaN
3,60af5b022fbbe97183553d36,473748,11712,Cage Free Large White Eggs,86,eggs,16,dairy eggs,13,1.0
4,60af5b022fbbe97183553d36,473748,13176,Bag of Organic Bananas,24,fresh fruits,4,produce,1,1.0


In [ ]:
replaceRoot = {
    "$replaceRoot": {"newRoot": {"_id": "$_id","order_id": "$order_id","product_name": "$products.product_name" }}}

df_with_replace2 = query(cll, [match, unwind, replaceRoot])
df_with_replace2.head()

In [118]:
# Unwinding the array field only divides the elements of the array into different records,
# but it does not make the attributes of such elements directly accessible
df_no_replace = query(cll, [match, unwind])
df_no_replace.head()

localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 691cb4e37e1d880034b755b2, topology_type: Single, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>


AttributeError: 'NoneType' object has no attribute 'head'

Use $mergeObjects when you want to combine entire objects.
Manually build the newRoot object when you want to pick and choose specific fields from the original document and its sub-documents.

### Exercise 2.2
##### Count the number of orders that contain at least 20 products

In [ ]:
match = {"$match": {"products": {"$type": "array"}}}
project = {"$project": {"order_id": 1, "n_products":{"$size":"$products"}}}
match1 = {"$match": {"n_products": {"$gte": 20}}}
group = {"$group":{"_id": None, "count": {"$sum": 1}}}

query(cll, [match, project, match1, group])

In [ ]:
# Answering the same query but in different manner
unwind = {"$unwind": "$products"}

group1 = {"$group": { "_id": "$_id","numberOfProducts": { "$sum": 1 }}}


match1 = {"$match": {"numberOfProducts": { "$gte": 20 }}}


group2 = {"$group": {"_id": None, "count": { "$sum": 1 }}}


query(cll, [unwind, group1, match1, group2])


In [ ]:
match = {"$match": {"products": {"$type": "array"}}}
project = {"$project": {"order_id": 1, "n_products":{"$max":"$products.add_to_cart_order"}}}
match1 = {"$match": {"n_products": {"$gte": 20}}}
group = {"$group":{"_id": None, "count": {"$sum": 1}}}

query(cll, [match, project, match1, group])

### Exercise 2.3
##### After using the ``$unwind`` operator on the products contained in order 473748, try to restore the original data.

In [ ]:
match = {"$match": {"order_id": 473748}}
unwind = {"$unwind": "$products"}
group = {"$group": {"_id": "$order_id", "products": { "$push": {"product_id": "$products.product_id",
                                                                "product_name": "$products.product_name",
                                                                "aisle_id": "$products.aisle_id",
                                                                "aisle": "$products.aisle",
                                                                "department_id": "$products.department_id",
                                                                "department": "$products.department",
                                                                "add_to_cart_order": "$products.add_to_cart_order",
                                                                "reordered": "$products.reordered"}}}}

query(cll, [match, unwind, group]).to_dict()

### Exercise 2.4
##### Select all order ids for user 1 and put them in a list

In [ ]:
match = {"$match": {"user_id": 1}}
group = {"$group": {"_id": "$user_id", "products": { "$push": "$order_id"}}}
project = {"$project": {"_id": 0}}

res = query(cll, [match, group, project])
res

In [ ]:
res.iloc[0,0]

### Exercise 2.5
##### String search with PyMongo: How many orders can produce _some sort_ of Caprese, i.e. how many orders contain both Tomatoes and Mozzarella?

The ``$text`` operator works only at the beginning of a pipeline, better use an alternative from Python library ``re``.
``re`` (with its variant ``regex``) is a Python library to perform operations with Regular Expressions, i.e. string patterns. On the web you can find several interfaces to practice with and test your regexs (e.g. https://regexr.com/).

In [ ]:
import re
import regex

In [ ]:
# Using $unwind to retrieve products that contain the word "tomato"
unwind = {"$unwind": "$products"}
group = {"$group": {"_id": "$products.product_name"}}
match = {"$match": {"_id": re.compile(".*tomato.*", re.IGNORECASE)}}
match1={"$match": {"id":re.compile(".*mozzarella.*", re.IGNORECASE)}}

query(cll, [unwind, group, match,match1])

In [ ]:
# Using $elemMatch to retrieve the orders that contain products whose name contains the word "mozzarella" and products that contain the word "tomato"
match = {
    "$match": {"$and":
                        [{"products":{"$elemMatch": {"product_name": re.compile(".*mozzarella.*", re.IGNORECASE)}}},
                         {"products":{"$elemMatch": {"product_name": re.compile(".*tomato.*", re.IGNORECASE)}}}]
    }
}

res = query(cll, [match])
res.head()


#This is the most important tool here. $elemMatch (short for "Element Match") lets you peek inside an array (our products list) and check if at least one item inside matches your criteria.

In [ ]:
# Check
[d['product_name'] for d in res.loc[0,"products"]]

In [ ]:
# How many
res.shape[0]

### Exercise 2.6
##### On average, how many products does an order include?

In [ ]:
unwind = {"$unwind":"$products"}
group1 = {"$group":{"_id":"$order_id", "count":{"$sum":1}}}
group2 = {"$group":{"_id":None, "avg":{"$avg":"$count"}}}

query(cll, [unwind, group1, group2])

In [ ]:
# Solution 1
match = {"$match": {"products": {"$type": "array"}}}
project = {"$project": {"n_products": {"$size": "$products"}}}
group = {"$group": {"_id": None, "avg_products": {"$avg": "$n_products"}}}

query(cll, [match, project, group])

In [ ]:
# Solution 3
unwind = {'$unwind': '$products'}
group = {"$group" : {"_id" : "$order_id", "tot_products" : { "$sum": 1}}}
group1 = {"$group" : {"_id" : None, "avg_products": { "$avg": '$tot_products'}}}

query(cll, [unwind, group, group1])

### Exercise 2.7

##### "Spirits and diapers". 
##### According to a traditional example in microeconomics, spirits and products for babies are complement products (positively correlated). 
##### Get orders that contain at least one item of aisle "spirits" and one of aisle "diapers wipes" or "baby accessories" in the first 5 purchased items

In [ ]:
match = {
    "$match": {
        "$and": [
            {"products":{"$elemMatch": {"aisle":"spirits", "add_to_cart_order": {"$lte": 5}}}},
            {"$or": [
                {"products": {"$elemMatch": {"aisle":"diapers wipes", "add_to_cart_order": {"$lte": 5}}}},
                {"products": {"$elemMatch": {"aisle":"baby accessories", "add_to_cart_order": {"$lte": 5}}}}
            ]}
        ]
    }
}
project = {"$project":{"_id":"$order_id", "all_aisles":"$products.aisle"}}
sort = {"$sort":{"_id":1}}

query(cll, [match, project, sort])

In [ ]:
unwind = {"$unwind": "$products"}
match = {"$match": {"products.add_to_cart_order": {"$lte": 5}}}
group = {"$group": {"_id": "$order_id", "first_five_aisles": {"$push": "$products.aisle"}}}
match1 = {
    "$match": {
        "$or": [{"first_five_aisles": "diapers wipes"}, {"first_five_aisles": "baby accessories"}], 
        "first_five_aisles": "spirits"
    }
}
sort = {"$sort":{"_id":1}}

query(cll, [unwind, match, group, match1, sort])

### Exercise 2.8
##### What's the most ordered product in each hour of the day?

In [ ]:
unwind = {"$unwind": "$products"}
group = {"$group": {"_id":{"product_name": "$products.product_name","hour": "$order_hour_of_day"}, "n_products":{"$sum":1}}}
sort = {"$sort": {"n_products": -1}}
group1 = {"$group": {"_id": "$_id.hour", "product_name": {"$first": "$_id.product_name"}, "count": {"$first": "$n_products"}}}
sort1 = {"$sort": {"_id": 1}}

In [ ]:
query(cll, [unwind, group, sort, group1, sort1])

In [ ]:
# Check this result is in line with the overall number of purchases for each product
unwind = {"$unwind": "$products"}
group = {"$group": {"_id":"$products.product_name", "n_products":{"$sum":1}}}
sort = {"$sort":{"n_products":-1}}

query(cll, [unwind, group, sort])

### Exercise 2.9
##### Which aisle do customers visit first? Count the number of times each aisle is visited as first.

In [ ]:

unwind = {"$unwind":"$products"}
match = {"$match":{"products.add_to_cart_order": 1}}
group = {"$group":{"_id":"$products.aisle", "count":{"$sum":1}}}
sort = {"$sort":{"count":-1}}

query(cll, [unwind, match, group, sort])

### Exercise 2.10
##### Which aisle do customers visit last? Count the number of times each aisle is visited as last.

In [ ]:
unwind = {"$unwind": "$products"}
sort = {"$sort": {"products.add_to_cart_order": -1}}
group = {"$group": {"_id": "$order_id", "aisle": {"$first":"$products.aisle"}}}
group1 = {"$group": {"_id": "$aisle", "count": {"$sum": 1}}}
sort = {"$sort": {"count": -1}}

query(cll, [unwind, sort, group, group1, sort])

### Exercise 2.11
##### Find the average position in which each department is visited (for the first time in an order).

In [ ]:
unwind = {"$unwind": "$products"}
group = {"$group": {"_id":["$order_id", "$products.department"], "department": {"$first":"$products.department"}, "first_time": {"$min":"$products.add_to_cart_order"}}}
group1 = {"$group": {"_id": "$department", "avg_first_time":{"$avg":"$first_time"}}}
sort = {"$sort": {"avg_first_time": 1}}
         
query(cll, [unwind,group,group1,sort])




In [ ]:
unwind = {"$unwind": "$products"}
group = {"$group": {"_id":["$order_id", "$products.department"], "department": {"$first":"$products.department"}, "first_time": {"$min":"$products.add_to_cart_order"}}}
group1 = {"$group": {"_id": "$department", "avg_first_time":{"$avg":"$first_time"}}}
sort = {"$sort": {"avg_first_time": 1}}

query(cll, [unwind, group, group1, sort])

##### Retrieve the top 10 departments by number of products ordered.

In [ ]:
unwind = {"$unwind": "$products"}
group = {"$group": {"_id": "$products.department", "count": {"$sum":1}}}
sort = {"$sort": {"count": -1}}
limit = {"$limit": 10}

query(cll, [unwind, group, sort, limit])

### Exercise 2.2

##### Which are the top 10 most purchased products?

In [ ]:
unwind = {"$unwind": "$products"}
group = {"$group": {"_id": "$products.product_name", "count": {"$sum":1}}}
sort = {"$sort": {"count": -1}}
limit = {"$limit": 10}

query(cll, [unwind, group, sort, limit])

### Exercise 2.6
##### On average, how many different products are purchased in a single aisle?

In [ ]:
unwind = {"$unwind":"$products"}
group = {"$group": {"_id":{"aisle":"$products.aisle", "product":"$products.product_id"}}}
group1 = {"$group": {"_id": "$_id.aisle", "products_per_aisle":{"$sum":1}}}
group2 = {"$group": {"_id": None, "avg_products_per_aisle":{"$avg":"$products_per_aisle"}}}

query(cll, [unwind, group, group1, group2])

In [ ]:
group = {"$group": {"_id": "$user_id", "count":{"$sum": 1}}}
group1 = {"$group": {"_id": None, "avg": {"$avg": "count"}}}

### Exercise 2.8
##### Get the Top 10 orders by number of departments involved

In [ ]:
unwind = {"$unwind":"$products"}
group = {"$group": {"_id": {"order":"$order_id", "department":"$products.department"}}}
group1 = {"$group":{"_id":"$_id.order", "departments_per_order":{"$sum":1}}}
sort = {"$sort": {"departments_per_order":-1}}
limit = {"$limit": 10}

query(cll, [unwind, group, group1, sort, limit])

### Exercise 2.10
##### How many aisles does each department have?

In [ ]:
unwind = {"$unwind":"$products"}
group = {"$group": {"_id":{"department": "$products.department", "aisle": "$products.aisle"}}}
group1 = {"$group": {"_id": "$_id.department", "n_aisles":{"$sum":1}}}

query(cll, [unwind, group, group1])

In [ ]:
# Qualitative check
group2 = {"$group": {"_id": "$_id.department", "aisles":{"$push":"$_id.aisle"}}}

query(cll, [unwind, group, group2])

In [ ]:
# Quantitative check
df1 = query(cll, [unwind, group, group1]).set_index("_id")
df2 = query(cll, [unwind, group, group2]).set_index("_id")

df2['n_aisles'] = df2['aisles'].apply(len)
pd.merge(df1, df2, left_index = True, right_index = True)

### Exercise 2.11
##### Are there any cart order patterns?
1. <b>Find the name of the top-20 most ordered products as the first item in a single order</b></br>
2. <b>Find the name of the top-20 most ordered products as the third item in a single order</b></br>

In [ ]:
# 1.
unwind = {"$unwind":"$products"}
match_first = {"$match": {"products.add_to_cart_order": 1}}
group = {"$group": {"_id":"$products.product_name", "n_purchases":{"$sum": 1}}}
sort = {"$sort": {"n_purchases":-1}}
limit = {"$limit": 20}

query(cll, [unwind, match_first, group, sort, limit])

In [ ]:
# 2.
unwind = {"$unwind":"$products"}
match_third = {"$match": {"products.add_to_cart_order": 3}}
group = {"$group": {"_id":"$products.product_name", "n_purchases":{"$sum": 1}}}
sort = {"$sort": {"n_purchases":-1}}
limit = {"$limit": 20}

query(cll, [unwind, match_third, group, sort, limit])